In [1]:
import pandas as pd
import numpy as np

In [2]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

In [3]:
train_df.tail()

,Unnamed: 0,paper_id,title,primary_category,authors,first_author,abstract,days_since_submission
136233,136233,abs-2408.08541v1,Where is the signal in tokenization space?,cs.CL,Renato Lui Geh; Honghua Zhang; Kareem Ahmed; B...,Renato Lui Geh,Large Language Models (LLMs) are typically shi...,668
136234,136234,abs-2408.08564v1,Collaborative Cross-modal Fusion with Large La...,cs.IR,Zhongzhou Liu; Hao Zhang; Kuicai Dong; Yuan Fang,Zhongzhou Liu,Despite the success of conventional collaborat...,668
136235,136235,abs-2408.08624v1,RealMedQA: A pilot biomedical question answeri...,cs.CL,Gregory Kell; Angus Roberts; Serge Umansky; Yu...,Gregory Kell,Clinical question answering systems have the p...,668
136236,136236,abs-2408.08648v1,Understanding Enthymemes in Argument Maps: Bri...,cs.AI,Jonathan Ben-Naim; Victor David; Anthony Hunter,Jonathan Ben-Naim,Argument mining is natural language processing...,668
136237,136237,abs-2408.08651v2,Reasoning Beyond Bias: A Study on Counterfactu...,cs.CL,Kyle Moore; Jesse Roberts; Thao Pham; Douglas ...,Kyle Moore,Language models are known to absorb biases fro...,647


In [29]:
test_df.head()

,Unnamed: 0,paper_id,title,primary_category,authors,first_author,abstract,days_since_submission
0,0,2604.26951v1,Turning the TIDE: Cross-Architecture Distillat...,cs.CL,Gongbo Zhang; Wen Wang; Ye Tian; Li Yuan,Gongbo Zhang,Diffusion large language models (dLLMs) offer ...,47
1,1,2604.26231v1,ProMax: Exploring the Potential of LLM-derived...,cs.IR,Yi Zhang; Yiwen Zhang; Kai Zheng; Tong Chen; H...,Yi Zhang,The remarkable text understanding and generati...,47
2,2,2604.26567v1,AirZoo: A Unified Large-Scale Dataset for Grou...,cs.CV,Xiaoya Cheng; Rouwan Wu; Xinyi Liu; Zeyu Cui; ...,Xiaoya Cheng,Despite the rapid progress in data-driven 3D v...,47
3,3,2604.26565v1,"DenseStep2M: A Scalable, Training-Free Pipelin...",cs.CV,Mingji Ge; Qirui Chen; Zeqian Li; Weidi Xie,Mingji Ge,Long-term video understanding requires interpr...,47
4,4,2604.26520v1,3D-LENS: A 3D Lifting-based Elevated Novel-vie...,cs.CV,William Grolleau; Astrid Sabourin; Guillaume L...,William Grolleau,Aerial-Ground Re-Identification (AG-ReID) is c...,47


In [ ]:
# !pip install nltk
# !pip install scikit-learn

In [30]:
def create_data_input(df, columns):
    result=[]
    for id, row in df.iterrows():
        tmp_text = [col+' : '+str(row[col]) for col in columns]
        text = ' || '.join(tmp_text)
        result.append(text)
    return result

In [31]:
tfidf_input = create_data_input(train_df, columns = ['title', 'abstract', 'authors', 'primary_category'])
print(len(tfidf_input))

136238


In [32]:
print(tfidf_input[2])

title : An Empirical Analysis of Search in GSAT || abstract : We describe an extensive study of search in GSAT, an approximation procedure
for propositional satisfiability. GSAT performs greedy hill-climbing on the
number of satisfied clauses in a truth assignment. Our experiments provide a
more complete picture of GSAT's search than previous accounts. We describe in
detail the two phases of search: rapid hill-climbing followed by a long plateau
search. We demonstrate that when applied to randomly generated 3SAT problems,
there is a very simple scaling with problem size for both the mean number of
satisfied clauses and the mean branching rate. Our results allow us to make
detailed numerical conjectures about the length of the hill-climbing phase, the
average gradient of this phase, and to conjecture that both the average score
and average branching rate decay exponentially during plateau search. We end by
showing how these results can be used to direct future theoretical analysis.
This

In [33]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

class TextPreprocessor:
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()
        self.vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2), max_df=0.9, min_df = 10)

    def preprocess(self, text):
        # Lowercase
        text = text.lower()
        # Remove punctuation and numbers
        text = re.sub(r'[^a-z\s]', '', text)
        # Tokenize
        tokens = word_tokenize(text)
        # Remove stop words and lemmatize
        tokens = [self.lemmatizer.lemmatize(word) for word in tokens if word not in self.stop_words]
        return ' '.join(tokens)

    def fit_transform(self, texts):
        preprocessed_texts = [self.preprocess(text) for text in texts]
        return self.vectorizer.fit_transform(preprocessed_texts)

    def transform(self, texts):
        preprocessed_texts = [self.preprocess(text) for text in texts]
        return self.vectorizer.transform(preprocessed_texts)
    
preprocessor = TextPreprocessor()
tfidf_matrix = preprocessor.fit_transform(tfidf_input)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\TTD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\TTD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\TTD\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\TTD\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\TTD\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [35]:
import scipy.sparse
import joblib

scipy.sparse.save_npz('tfidf_matrix.npz', tfidf_matrix)
joblib.dump(preprocessor, 'text_preprocessor.pkl')

['text_preprocessor.pkl']

In [36]:
tfidf_matrix.shape

(136238, 10000)

In [37]:
from sklearn.metrics.pairwise import cosine_similarity as cossim
pair = (0, 0)
max = -1.0
j=3
for i in range(0, 7700):
    if max < cossim(tfidf_matrix[j], tfidf_matrix[i])[0][0] and i!=j:
        max = cossim(tfidf_matrix[j], tfidf_matrix[i])[0][0]
        pair = (j, i)
print(max, pair)
    

0.4019697906795165 (3, 2369)


In [38]:
print(cossim(tfidf_matrix[7], tfidf_matrix[2]))

[[0.02958113]]


In [39]:
print(tfidf_matrix[3])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 81 stored elements and shape (1, 10000)>
  Coords	Values
  (0, 5875)	0.0469463571719604
  (0, 5306)	0.02296908165101558
  (0, 6972)	0.028566636930094147
  (0, 6410)	0.024668328855197694
  (0, 8389)	0.08332927261835546
  (0, 2438)	0.12443508506778865
  (0, 8927)	0.03851569671855645
  (0, 600)	0.02650663760325941
  (0, 1955)	0.044160396851041846
  (0, 6917)	0.044160396851041846
  (0, 8800)	0.035146382045248434
  (0, 2689)	0.04465935393262145
  (0, 506)	0.03682365396421173
  (0, 7008)	0.055867092699381886
  (0, 1397)	0.18792652639139662
  (0, 5111)	0.044008897595442124
  (0, 4784)	0.11928487296677574
  (0, 4996)	0.27040682935201593
  (0, 7038)	0.33891760632004514
  (0, 1997)	0.6323078965311981
  (0, 7328)	0.047785795297180717
  (0, 6096)	0.09387454070492952
  (0, 9523)	0.03561005482619156
  (0, 10)	0.05094663621721399
  (0, 2138)	0.06396428598100014
  :	:
  (0, 686)	0.07710644975072392
  (0, 5108)	0.0605023054653859
  (0, 8153)

In [40]:
print(preprocessor.vectorizer.get_feature_names_out()[400::700])

['algorithm paper' 'boost' 'control' 'discussing' 'failed' 'higher level'
 'key component' 'metadata' 'new metric' 'plate' 'recognition author'
 'set new' 'system' 'unstable']
